In this notebook, let's see how to handle missing data on the house price prediction dataset.

# **Importing necessary libraries**

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

# **Loading the datasets**

In [ ]:
train=pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test=pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
train.shape

In [ ]:
test.shape

In [ ]:
combined_df=pd.concat([train.drop('SalePrice',axis=1),test]).reset_index(drop=True)
combined_df

# **Missing Values**

Let's see the columns which have missing values in them

In [ ]:
for col in combined_df:
    if combined_df[col].isnull().sum()>0:
        print(col,combined_df[col].isnull().sum())

Let's see the percentage of missingness in these columns

In [ ]:
missing_cols=[]

In [ ]:
for col in combined_df:
    if combined_df[col].isnull().sum()>0:
        missing_cols.append(col)
        print(col,(combined_df[col].isnull().sum()/len(combined_df))*100)

In [ ]:
missing_cols=pd.DataFrame(missing_cols)

In [ ]:
missing_cols

In [ ]:
import missingno as msno
msno.matrix(combined_df)

# **Removing rows**

Let's drop rows with null values for columns that are MCAR and missing percentage is less than 0.1

In [ ]:
drop_cols=['Utilities','Electrical','KitchenQual','Functional','SaleType']
combined_df.dropna(subset=drop_cols,inplace=True)

In [ ]:
combined_df['Utilities'].isnull().sum()

# **Dropping columns**

Let's drop columns with more than 90% missing values

In [ ]:
drop_cols=['Alley','PoolQC','MiscFeature']
combined_df.drop(drop_cols,axis=1,inplace=True)
combined_df.shape

# **Filling nulls with constants**

Columns like GarageQual, GarageType have missing values because there's no garage, so for these type of columns, we can fill the missing values with "Missing" for categorical and 0 for numerical

In [ ]:
cat_cols=["BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","FireplaceQu","GarageType","GarageFinish","GarageQual","GarageCond","Fence"]
num_cols=["BsmtFinSF1","BsmtFinSF2","BsmtUnfSF","TotalBsmtSF","BsmtFullBath","BsmtHalfBath","GarageYrBlt","GarageCars","GarageArea"]

Using SimpleImputer to fill missing values with constants ("missing" for categorical, 0 for numerical)

In [ ]:
from sklearn.impute import SimpleImputer
si_cat=SimpleImputer(fill_value="missing",strategy='constant')
si_num=SimpleImputer(fill_value=0,strategy='constant')

In [ ]:
combined_df[cat_cols]=si_cat.fit_transform(combined_df[cat_cols])
combined_df[cat_cols].isnull().sum()

In [ ]:
combined_df[num_cols]=si_num.fit_transform(combined_df[num_cols])
combined_df[num_cols].isnull().sum()

# **Filling nulls with median and mode**

Let's check the skewness of 'MasVnrArea' column

In [ ]:
combined_df['MasVnrArea'].skew()

Using SimpleImputer to fill remaining categorical columns('Exterior1st','Exterior2nd','MasVnrType') with mode and 'MasVnrArea' with median( since skewness > 1)

In [ ]:
cat_cols=['Exterior1st','Exterior2nd','MasVnrType']
num_cols=['MasVnrArea']
from sklearn.impute import SimpleImputer
si_cat=SimpleImputer(strategy='most_frequent')
si_num=SimpleImputer(strategy='median')

In [ ]:
combined_df[cat_cols]=si_cat.fit_transform(combined_df[cat_cols])
combined_df[cat_cols].isnull().sum()

In [ ]:
combined_df[num_cols]=si_num.fit_transform(combined_df[num_cols])
combined_df[num_cols].isnull().sum()

# **Filling nulls with KNNImputer**

Using KNNImputer to fill the missing values of 'LotFrontage' column

In [ ]:
from sklearn.impute import KNNImputer
ki=KNNImputer(n_neighbors=5)
combined_df['LotFrontage']=ki.fit_transform(combined_df[['LotFrontage']])

In [ ]:
combined_df['LotFrontage'].isnull().sum()